In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, lit
from pyspark.sql.functions import to_timestamp
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, col

spark = SparkSession.builder \
    .master("local[*]") \
    .appName("blue-owls-assessment") \
    .getOrCreate()

spark.version  # confirm Spark is running

'3.5.0'

In [17]:

order_df = spark.read.option("header", True).option("inferSchema",True).csv("/home/jovyan/blue-owls-de-assessment/submission/output/bronze/orders.csv")

df = order_df \
    .withColumn("customer_id", when(col("customer_id").isNull(), lit("UNKNOWN")).otherwise(col("customer_id"))) \
    .withColumn("order_id", when(col("order_id").isNull(), lit("UNKNOWN")).otherwise(col("order_id"))) \
    .withColumn("order_approved_at", to_timestamp("order_approved_at")) \
    .withColumn("order_delivered_carrier_date", to_timestamp("order_delivered_carrier_date")) \
    .withColumn("order_delivered_customer_date", to_timestamp("order_delivered_customer_date")) \
    .withColumn("order_estimated_delivery_date", to_timestamp("order_estimated_delivery_date")) \
    .withColumn("order_purchase_timestamp", to_timestamp("order_purchase_timestamp"))
df = df.dropDuplicates()
df = df.withColumn("_is_valid",((col("order_delivered_customer_date") > col("order_purchase_timestamp"))))
print(df.count())
df = df.filter(
    col("order_purchase_timestamp") >= "2018-07-01"
)
print(df.count())
window = Window.partitionBy("order_id").orderBy(col("_ingested_at").desc())
df = df.withColumn("rn", row_number().over(window)) \
       .filter("rn = 1") \
       .drop("rn")

df.coalesce(1).write.format("csv").mode("overwrite").option("header",True).save("/home/jovyan/blue-owls-de-assessment/submission/output/silver/orders.csv")

99000
12767


In [26]:
order_items_df = spark.read.option("header", True).option("inferSchema",True).csv("/home/jovyan/blue-owls-de-assessment/submission/output/bronze/order_items.csv")

df1= order_items_df \
    .withColumn("order_id", when(col("order_id").isNull(), lit("UNKNOWN")).otherwise(col("order_id"))) \
    .withColumn("product_id", when(col("product_id").isNull(), lit("UNKNOWN")).otherwise(col("product_id"))) \
    .withColumn("seller_id", when(col("seller_id").isNull(), lit("UNKNOWN")).otherwise(col("seller_id"))) \
    .withColumn("freight_value", col("freight_value").cast("double"))\
    .withColumn("price", col("price").cast("double")) \
    .withColumn("shipping_limit_date", to_timestamp("shipping_limit_date")) \
    .withColumn("_ingested_at", to_timestamp("_ingested_at"))

df1 = df1.withColumn("_is_valid",((col("price") >= 0)))
print(df1.count())
window = Window.partitionBy("order_id").orderBy(col("_ingested_at").desc())
df1 = df1.withColumn("rn", row_number().over(window)) \
       .filter("rn = 1") \
       .drop("rn")
print(df1.count())
df1.coalesce(1).write.format("csv").mode("overwrite").option("header",True).save("/home/jovyan/blue-owls-de-assessment/submission/output/silver/order_items.csv")


112000
98087


In [25]:
customers_df = spark.read.option("header", True).option("inferSchema",True).csv("/home/jovyan/blue-owls-de-assessment/submission/output/bronze/customers.csv")

df2= customers_df \
     .withColumn("customer_city", when(col("customer_city").isNull(), lit("UNKNOWN")).otherwise(col("customer_city"))) \
     .withColumn("customer_id", when(col("customer_id").isNull(), lit("UNKNOWN")).otherwise(col("customer_id"))) \
     .withColumn("customer_state", when(col("customer_state").isNull(), lit("UNKNOWN")).otherwise(col("customer_state"))) \
     .withColumn("customer_unique_id", when(col("customer_unique_id").isNull(), lit("UNKNOWN")).otherwise(col("customer_unique_id"))) \
     .withColumn("customer_zip_code_prefix", when(col("customer_zip_code_prefix").isNull(), lit("UNKNOWN")).otherwise(col("customer_zip_code_prefix"))) \
     .withColumn("_ingested_at", to_timestamp("_ingested_at"))

print(df2.count())
window = Window.partitionBy("customer_unique_id").orderBy(col("_ingested_at").desc())
df2 = df2.withColumn("rn", row_number().over(window)) \
       .filter("rn = 1") \
       .drop("rn")
print(df2.count())
df2.coalesce(1).write.format("csv").mode("overwrite").option("header",True).save("/home/jovyan/blue-owls-de-assessment/submission/output/silver/customers.csv")

     


99000
95688


In [24]:
products_df = spark.read.option("header", True).option("inferSchema",True).csv("/home/jovyan/blue-owls-de-assessment/submission/output/bronze/products.csv")

df3 = products_df \
    .withColumn("product_description_lenght", col("product_description_lenght").cast("int")) \
    .withColumn("product_name_lenght", col("product_name_lenght").cast("int")) \
    .withColumn("product_photos_qty", col("product_photos_qty").cast("int")) \
    .withColumn("product_weight_g", col("product_weight_g").cast("int")) \
    .withColumn("product_height_cm", col("product_height_cm").cast("int")) \
    .withColumn("product_width_cm", col("product_width_cm").cast("int")) \
    .withColumn("product_length_cm", col("product_length_cm").cast("int")) \
    .withColumn(
        "product_category_name",
        when(col("product_category_name").isNull(), lit("UNKNOWN"))
        .otherwise(col("product_category_name"))
    ) \
    .withColumn(
        "product_photos_qty",
        when(col("product_photos_qty").isNull(), lit(0))
        .otherwise(col("product_photos_qty"))
    )

df3 = df3.withColumn(
    "_is_valid",
    (
        col("product_id").isNotNull() & 
        (col("product_weight_g").isNull() | (col("product_weight_g") >= 0)) &
        
        (col("product_height_cm").isNull() | (col("product_height_cm") >= 0)) &
        (col("product_width_cm").isNull() | (col("product_width_cm") >= 0)) &
        (col("product_length_cm").isNull() | (col("product_length_cm") >= 0))
    )
)
print(df3.count())
window = Window.partitionBy("product_id").orderBy(col("_ingested_at").desc())

df3 = df3.withColumn("rn", row_number().over(window)) \
       .filter("rn = 1") \
       .drop("rn")
print(df3.count())
df3.coalesce(1).write.format("csv").mode("overwrite").option("header",True).save("/home/jovyan/blue-owls-de-assessment/submission/output/silver/products.csv")

32000
32000


In [23]:
sellers_df = spark.read.option("header", True).option("inferSchema",True).csv("/home/jovyan/blue-owls-de-assessment/submission/output/bronze/sellers.csv")

df4 = sellers_df \
    .withColumn("seller_zip_code_prefix", col("seller_zip_code_prefix").cast("int")) \
    .withColumn(
        "seller_city",
        when(col("seller_city").isNull(), lit("UNKNOWN"))
        .otherwise(col("seller_city"))
    ) \
    .withColumn(
        "seller_state",
        when(col("seller_state").isNull(), lit("NA"))
        .otherwise(col("seller_state"))
    )
df4 = df4.withColumn(
    "_is_valid",
    (
        col("seller_id").isNotNull() & 
        (col("seller_zip_code_prefix").isNull() | (col("seller_zip_code_prefix") > 0))
    )
)
print(df4.count())
window = Window.partitionBy("seller_id").orderBy(col("_ingested_at").desc())

df4 = df4.withColumn("rn", row_number().over(window)) \
       .filter("rn = 1") \
       .drop("rn")
print(df4.count())
df4.coalesce(1).write.format("csv").mode("overwrite").option("header",True).save("/home/jovyan/blue-owls-de-assessment/submission/output/silver/sellers.csv")

3000
3000


In [22]:
payments_df = spark.read.option("header", True).option("inferSchema",True).csv("/home/jovyan/blue-owls-de-assessment/submission/output/bronze/payments.csv")

df5 = payments_df \
    .withColumn("payment_installments", col("payment_installments").cast("int")) \
    .withColumn("payment_sequential", col("payment_sequential").cast("int")) \
    .withColumn("payment_value", col("payment_value").cast("double")) \
    .withColumn(
        "payment_installments",
        when(col("payment_installments").isNull(), lit(1))
        .otherwise(col("payment_installments"))
    ) \
    .withColumn(
        "payment_type",
        when(col("payment_type").isNull(), lit("UNKNOWN"))
        .otherwise(col("payment_type"))
    )
df5 = df5.withColumn(
    "_is_valid",
    (
        col("order_id").isNotNull() &
        col("payment_sequential").isNotNull() &
        (col("payment_value").isNull() | (col("payment_value") >= 0)) &
        (col("payment_installments") >= 1)
    )
)

window = Window.partitionBy("order_id", "payment_sequential") \
               .orderBy(col("_ingested_at").desc())

df5 = df5.withColumn("rn", row_number().over(window)) \
       .filter("rn = 1") \
       .drop("rn")
df5.coalesce(1).write.format("csv").mode("overwrite").option("header",True).save("/home/jovyan/blue-owls-de-assessment/submission/output/silver/payments.csv")